# Hệ Thống CSKH Nhớ Lịch Sử Hội Thoại

Hệ thống CSKH cho một phòng khám tư nhân tại Việt Nam.

Hệ thống của chúng ta sẽ có:

- **Một**: Bộ nhớ dai dẳng qua SQLite — bệnh nhân gọi lại ngày hôm sau, bot vẫn nhớ.

- **Hai**: Tool Calling — bot có thể tra cứu lịch khám còn trống và "đặt lịch" (mô phỏng).

- **Ba**: Toàn bộ được giám sát qua LangSmith.

In [1]:
import os
import sqlite3
from datetime import datetime, timedelta
from typing import Annotated, TypedDict
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.sqlite import SqliteSaver

load_dotenv(override=True)

True

In [2]:
# Dữ liệu giả lập — trong thực tế sẽ kết nối database thật
LICH_KHAM_GIA_LAP = {
    "2026-04-11": ["09:00", "10:30", "14:00", "15:30"],
    "2026-04-12": ["08:30", "11:00", "13:30"],
    "2026-04-13": ["09:30", "10:00", "14:30", "16:00"],
    "2026-04-14": ["08:00", "11:30", "15:00"],
}
LICH_DA_DAT = {}  # Lưu tạm các lịch đã đặt

In [3]:
@tool
def tra_cuu_lich_trong(ngay: str) -> str:
    """
    Tra cứu các khung giờ còn trống để đặt lịch khám.
    Tham số ngay phải theo định dạng YYYY-MM-DD, ví dụ: 2026-04-11
    """
    if ngay not in LICH_KHAM_GIA_LAP:
        return f"Không có thông tin lịch cho ngày {ngay}. Vui lòng chọn ngày khác."
    
    gio_trong = LICH_KHAM_GIA_LAP[ngay]
    gio_da_dat = LICH_DA_DAT.get(ngay, [])
    gio_con_trong = [g for g in gio_trong if g not in gio_da_dat]

    if not gio_con_trong:
        return f"Ngày {ngay} đã hết lịch. Vui lòng chọn ngày khác."
    
    gio_list = ", ".join(gio_con_trong)
    return f"Ngày {ngay} còn các khung giờ: {gio_list}"

@tool
def dat_lich_kham(ten_benh_nhan: str, ngay: str, gio: str, ly_do: str) -> str:
    """
    Đặt lịch khám cho bệnh nhân.
    Tham số:
    - ten_benh_nhan: Họ tên đầy đủ
    - ngay: Ngày khám theo định dạng YYYY-MM-DD
    - gio: Giờ khám, ví dụ: 09:00
    - ly_do: Lý do khám hoặc triệu chứng chính
    """
    gio_trong = LICH_KHAM_GIA_LAP.get(ngay, [])
    gio_da_dat = LICH_DA_DAT.get(ngay, [])

    if gio not in gio_trong:
        return f"Khung giờ {gio} ngày {ngay} không có trong lịch làm việc."
    
    if gio in gio_da_dat:
        return f"Rất tiếc, khung giờ {gio} ngày {ngay} vừa có người đặt. Vui lòng chọn giờ khác."

    # Đặt lịch
    if ngay not in LICH_DA_DAT:
        LICH_DA_DAT[ngay] = []
    LICH_DA_DAT[ngay].append(gio)

    ma_lich = f"PK{ngay.replace('-', '')}{gio.replace(':', '')}"

    return (f"Đặt lịch thành công!\n"
            f"Mã lịch hẹn: {ma_lich}\n"
            f"Bệnh nhân: {ten_benh_nhan}\n"
            f"Ngày khám: {ngay} lúc {gio}\n"
            f"Lý do khám: {ly_do}\n"
            f"Vui lòng đến trước 15 phút và mang theo CMND/CCCD")

@tool
def xem_lich_hom_nay() -> str:
    """
    Xem lịch khám ngày hôm nay và ngày mai để tư vấn cho bệnh nhân.
    Không cần tham số đầu vào.
    """
    hom_nay = datetime.now().strftime("%Y-%m-%d")
    ngay_mai = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")

    ket_qua = []
    for ngay in [hom_nay, ngay_mai]:
        ten_ngay = "Hôm nay" if ngay == hom_nay else "Ngày mai"
        gio_trong = LICH_KHAM_GIA_LAP.get(ngay, [])
        gio_da_dat = LICH_DA_DAT.get(ngay, [])
        gio_con_trong = [g for g in gio_trong if g not in gio_da_dat]
        if gio_con_trong:
            ket_qua.append(f"{ten_ngay} ({ngay}): còn {len(gio_con_trong)} khung giờ trống")
        else: 
            ket_qua.append(f"{ten_ngay} ({ngay}): đã đầy lịch")

    return "\n".join(ket_qua)

# Tap hop tools
danh_sach_tools = [tra_cuu_lich_trong, dat_lich_kham, xem_lich_hom_nay]

In [4]:
# State
class CskhState(TypedDict):
    messages: Annotated[list, add_messages]

In [5]:
# Khởi tạo LLM và bind tools
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
llm_co_tools = llm.bind_tools(danh_sach_tools)

In [6]:
SYSTEM_PROMPT = """Bạn là trợ lý CSKH của Phòng Khám Đa Khoa Tâm An, TP.HCM.

Nhiệm vụ:
- Hỗ trợ bệnh nhân đặt lịch khám, tra cứu lịch trống
- Trả lời câu hỏi về dịch vụ của phòng khám
- Nhớ thông tin bệnh nhân trong suốt cuộc trò chuyện

Quy tắc giao tiếp:
- Xưng hô lịch sự, phù hợp (anh/chị/em)
- Luôn xác nhận lại thông tin trước khi đặt lịch (tên, ngày, giờ)
- Nếu bệnh nhân mô tả triệu chứng cấp tính (đau ngực, khó thở, xuất huyết), 
  hướng dẫn đến khoa Cấp Cứu ngay, không đặt lịch hẹn thông thường
- Trả lời ngắn gọn, rõ ràng — không quá 4-5 câu mỗi lần

Thông tin phòng khám:
- Địa chỉ: 123 Nguyễn Trãi, Q.1, TP.HCM
- Giờ làm việc: 8:00 - 17:00, Thứ 2 đến Thứ 7
- Hotline: 028-xxxx-xxxx"""


def node_cskh(state: CskhState) -> dict:
    hom_nay = datetime.now().strftime("%Y-%m-%d")

    system_dynamic = SYSTEM_PROMPT + f"""

Thời gian hiện tại:
- Hôm nay là: {hom_nay}
- Luôn suy luận chính xác các mốc như "hôm nay", "ngày mai"

Nếu không chắc chắn về ngày, hãy hỏi lại bệnh nhân.
"""

    tat_ca_tin_nhan = [SystemMessage(content=system_dynamic)] + state["messages"]

    phan_hoi = llm_co_tools.invoke(tat_ca_tin_nhan)

    return {"messages": [phan_hoi]}

In [7]:
graph_builder = StateGraph(CskhState)

graph_builder.add_node("cskh", node_cskh)
graph_builder.add_node("tools", ToolNode(tools=danh_sach_tools))

# Edges
graph_builder.add_edge(START, "cskh")
graph_builder.add_conditional_edges(
    "cskh",
    tools_condition
)

graph_builder.add_edge("tools", "cskh")

In [8]:
# Compile SQlite checkpointing
DB_PATH = "phongkham_memory.db"
ket_noi = sqlite3.connect(DB_PATH, check_same_thread=False)
checkpointer = SqliteSaver(ket_noi)

graph = checkpointer and graph_builder.compile(checkpointer=checkpointer)

print(f"Bộ nhớ lưu tại: {DB_PATH}")

Bộ nhớ lưu tại: phongkham_memory.db


In [11]:
import gradio as gr

# =========================
# Session helpers
# =========================
def tao_session_benh_nhan(ma_benh_nhan: str):
    return {"configurable": {"thread_id": ma_benh_nhan}}


# =========================
# Chat handler
# =========================
def chat_fn(message, history, ma_benh_nhan):
    if not ma_benh_nhan:
        ma_benh_nhan = "BN_DEMO"

    config = tao_session_benh_nhan(ma_benh_nhan)

    ket_qua = graph.invoke(
        {"messages": [HumanMessage(content=message)]},
        config=config
    )

    tra_loi = ket_qua["messages"][-1].content

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": tra_loi})

    return history, history, ""


# =========================
# Load history từ DB
# =========================
def load_history(ma_benh_nhan):
    print("Loading thread:", ma_benh_nhan)
    if not ma_benh_nhan:
        return []

    config = tao_session_benh_nhan(ma_benh_nhan)
    state = graph.get_state(config)
    print("State:", state)

    if not state:
        return []

    msgs = state.values.get("messages", [])

    chat_history = []
    last_user = None

    for msg in msgs:
        if isinstance(msg, HumanMessage):
            last_user = msg.content
        elif isinstance(msg, AIMessage) and last_user:
            chat_history.append((last_user, msg.content))
            last_user = None

    return chat_history


# =========================
# UI
# =========================
with gr.Blocks(title="Phòng Khám Tâm An AI") as demo:

    gr.Markdown("## 🏥 CSKH AI — Phòng Khám Tâm An")
    gr.Markdown("Đặt lịch khám, tra cứu lịch nhanh chóng.")

    with gr.Row():
        ma_bn = gr.Textbox(
            label="Mã bệnh nhân (thread_id)",
            placeholder="VD: BN001"
        )
        btn_load = gr.Button("🔄 Tải lịch sử")

    chatbot = gr.Chatbot(height=400)
    msg = gr.Textbox(placeholder="Nhập tin nhắn...")
    send_btn = gr.Button("Gửi")

    # State giữ history UI
    state = gr.State([])

    # Events

    send_btn.click(
        chat_fn,
        inputs=[msg, state, ma_bn],
        outputs=[chatbot, state, msg]
    )

    btn_load.click(
        fn=lambda ma: (load_history(ma), load_history(ma)),
        inputs=ma_bn,
        outputs=[chatbot, state]
    )


# =========================
# Run
# =========================
if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Loading thread: PK202604120830
State: StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': 'PK202604120830'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())
Loading thread: PK202604120830
State: StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': 'PK202604120830'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())
Loading thread: PK202604120830
State: StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': 'PK202604120830'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())
Loading thread: PK202604120830
State: StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': 'PK202604120830'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())
Loading thread: PK202604120830
State: StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': 'PK202604120830'}}, metadata=None, created_at=None, parent_con